# 29 · Total a table within groups against a pandas groupby

**Question.** The South London contingency table is six numbers: the populations and the cholera
deaths of 32 sub-districts, totalled within the three water-supply categories Snow's Table XII
declares. Until 2026-10-26 those six were computed by hand and inlined into the worked example.
The application computes them now — so does `groupby(...).sum()` get the same six?

The sums are integers and nothing is approximated, so the tolerance is **zero**, not small.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

Six totals, recomputed from the fixture's input table and the columns it declares as kept.
`sub_district` is the column dropped; 64 rows collapse to 6.


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_29_group_table import run

report = run(Path.cwd() / "fixtures" / "group-snow-001.json")
print("agrees:", report["agrees"])

whole = report["comparisons"][0]
print(f"\n{whole['rowsIn']} rows -> {whole['groupsOut']} groups, dropping {whole['dropped']}\n")
for t in whole["totals"]:
    print(f"  {'OK ' if t['agrees'] else 'DIFF'} {t['pandas']:>7}  {' · '.join(t['group'])}")
for name, ok in whole["checks"].items():
    print(f"    {'OK ' if ok else 'DIFF'} {name}")


## The case worth the check: one empty cell

Every long table in Fieldwork can hold an empty cell, and summing over one silently returns a
total that is **too small by an unknown amount** — a stratum whose population nobody recorded
coming out smaller rather than coming out unknown. Fieldwork returns *nothing known* for that
group and names it.

**pandas has no option for that rule**, which this check found by reaching for the obvious one and
watching it not work. `min_count=1` reads as *"at least one value required"* and is the first
thing anyone writes here. It bounds how many values must be **present**, not whether any are
**absent** — so on a group holding eleven numbers and one empty cell it returns the eleven: the
default's answer, from the option named for missing data.

| expression | on the emptied group | what it is |
| --- | --- | --- |
| `sum()` | 147,945 | the default. Short by one sub-district, and nothing says so |
| `sum(min_count=1)` | 147,945 | the trap. Reads like the rule, behaves like the default |
| `sum(min_count=12)` | nothing known | right here, wrong in general: it also nulls any group with fewer than twelve rows |
| `sum(skipna=False)` per group | nothing known | **the comparison.** Nothing known exactly where a cell was empty, and the group size does not enter it |


In [ ]:
holed = report["comparisons"][1]
e, hidden = holed["emptied"], holed["whatTheDefaultWouldHide"]

print("emptied cell :", e["key"], "was", e["wasValue"])
print("group        :", " · ".join(holed["group"]))
print()
print(f"  Fieldwork                   {holed['fieldwork']}")
print(f"  pandas sum(skipna=False)    {holed['pandasSkipnaFalse']}   <- the comparison")
print(f"  pandas sum()                {holed['pandasDefault']}")
print(f"  pandas sum(min_count=1)     {holed['pandasMinCount1']}   <- not the rule")
print(f"  pandas sum(min_count=12)    {holed['pandasMinCountGroupSize']}   <- right by accident")
print()
print(holed["whyNotMinCount"])
print()
print(f"What the default would hide: {hidden['total']} instead of {hidden['insteadOf']} "
      f"{hidden['unit']} -- short by {hidden['shortBy']}.")
print(hidden["note"])
print()
print("gap reported:", holed["gaps"])
for name, ok in holed["checks"].items():
    print(f"  {'OK ' if ok else 'DIFF'} {name}")


## What this cannot establish

**That summing is right for these columns.** It is right for counts and wrong for rates — deaths
per 1,000 in three sub-districts do not add to deaths per 3,000 in one — and neither
implementation can tell which it has been given. Fieldwork offers sum and nothing else for that
reason, leaving `MEANS` unbuilt rather than offering a choice it cannot police. The protection is
the **absence of the other option**, which no check can verify.

**That nothing known is the right answer rather than the honest one.** It is a refusal to guess.
A practitioner who needs the number still has to go and find the missing cell.

**That the three supply categories are the right strata.** The both-companies group is 300,149 of
486,936 people, and what to do with it is the practitioner's judgement.


In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
